# Logistic Regression: Tuned Model (GridSearchCV) and SGD Variant

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

This notebook builds on the baseline of notebook 02 (same 40 features, same random 80/20 split) and asks whether tuning helps. It:

1. tunes a logistic regression (`liblinear` solver) with `GridSearchCV` over the penalty type (L1/L2) and the regularization strength `C`, using 5-fold cross-validation on the training data and ROC-AUC as selection metric;
2. evaluates the tuned model on the test set (metrics, confusion matrix, ROC and precision-recall curves);
3. compares it with a logistic regression trained by **stochastic gradient descent** (`SGDClassifier` with log loss);
4. shows, for illustration, how the decision threshold changes the F1-score;
5. inspects the largest standardized coefficients and the error types;
6. saves the results locally.

**Naming:** the confusion matrix, ROC/PR curves and coefficients below belong to the *tuned liblinear* model. The SGD model only appears in the comparison table of Section 6.

## Workflow

Run notebook 01 first; it creates `normalized_data.csv`. The baseline of notebook 02 is the natural reference. The accompanying thesis describes these models in Sections 3.2 and 4.1.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation. This notebook reads `normalized_data.csv`, which notebook 01 creates in the project root. Generated files are written to `../outputs/` and are not committed.

## Important limitations

- **Random split on time-series data.** The split is a stratified *random* split of rows. Neighbouring time steps of a machine are very similar, so near-duplicates can end up in both training and test sets and the scores can be **optimistic**. A chronological or group-wise split is a stricter test.
- **Origin of the 40 features.** The 40 features are the 20 most negative and the 20 most positive Pearson correlations with `Label_03` from notebook 01 (Section 8). That ranking was computed on the whole dataset, including later test rows, so the test scores are slightly biased.
- **Class coding.** Class `1` = Production is the positive class and class `0` = Changeover.
- **Threshold tuning on the test set.** Section 7 picks the F1-optimal threshold on the *test* set. This is illustrative only; the resulting F1 is optimistic. An unbiased choice needs a separate validation set or cross-validated predictions.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt` (run from the project root). All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    PrecisionRecallDisplay, RocCurveDisplay, accuracy_score,
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../normalized_data.csv")   # created by notebook 01
OUTPUT_DIR = Path("../outputs")              # local results, not for upload

LABEL_COL = "Label_03"
SEED = 42
TEST_SIZE = 0.2
THRESHOLD = 0.5

## 2. Load the normalized data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Run notebook 01 first (it creates normalized_data.csv in the project root) "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

if LABEL_COL not in df.columns:
    raise KeyError(
        f"Target column {LABEL_COL!r} not found. "
        f"The file has {len(df.columns)} columns."
    )

## 3. Select the 40 features

The same fixed list of 40 machine signals as in notebook 02, taken from the correlation ranking of notebook 01. Missing columns are reported; the model then continues with the columns that are present.

In [ ]:
feature_list = [
    'Spindle_smoothed_speed',
    'Program_status',
    'Z-axis_position_WCS',
    'Magazine_9998_number_Place_2',
    'Door_open',
    'Z-axis_position_MCS',
    'Magazine_9998_position_Place_3',
    'Program_Idle',
    'Magazine_position_Place_9',
    'Tool_number_Magazine_Place_43',
    'Magazine_number_Place_6',
    'Magazine_number_Place_7',
    'Magazine_number_Place_5',
    'Magazine_9998_position_Place_1',
    'Tool_number_Magazine_Place_9',
    'Magazine_number_Place_8',
    'Tool_magazine_position_WCS',
    'Tool_magazine_position_MCS',
    'Magazine_position_Place_17',
    'Number_of_tools',
    'Feed_override',
    'Rapid_traverse_button',
    'C_Axis_Motor_temperature',
    'Y_Axis_Motor_temperature',
    'Door_status_locked',
    'Spindle_position_WCS',
    'Spindle_position_MCS',
    'X-axis_position_MCS',
    'Net_running_time_program',
    'X_Axis_motor_temperature',
    'B_Axis_Motor_temperature',
    'Program_line_number',
    'Tool_number_Magazine_9998_Place_1',
    'X-axis_position_WCS',
    'Operation_mode',
    'LED_Chip_flushing',
    'Spindle_angular_velocity',
    'Spindle_speed',
    'Spindle_smoothed_modulation_depth',
    'LED_Coolant',
]

missing_features = [c for c in feature_list if c not in df.columns]
if missing_features:
    print("Warning: these features are missing and will be skipped:", missing_features)

feature_list = [c for c in feature_list if c in df.columns]
if not feature_list:
    raise ValueError("None of the expected features were found in the data.")
print(f"Using {len(feature_list)} features")

## 4. Build the feature matrix and split the data

Infinite values become missing values; missing values are filled later inside the model pipeline with medians learned from the training rows only. The split is stratified and identical to notebook 02 (same seed).

In [ ]:
X = df[feature_list].replace([np.inf, -np.inf], np.nan)
y = df[LABEL_COL].astype(int).to_numpy()

if set(np.unique(y)) != {0, 1}:
    raise ValueError(
        f"{LABEL_COL} must contain exactly the classes 0 and 1, "
        f"found: {np.unique(y).tolist()}"
    )

print(f"Missing / infinite feature values: {int(X.isna().sum().sum()):,}")
print(f"Share of class 1: {y.mean():.1%}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 5. Tune the logistic regression

The pipeline imputes, standardizes and fits a logistic regression with `class_weight="balanced"`. `GridSearchCV` tries 2 penalties × 4 values of `C` with stratified 5-fold cross-validation on the **training** data and keeps the combination with the best ROC-AUC. A larger `C` means weaker regularization; L1 can set coefficients exactly to zero.

In [ ]:
base = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        solver="liblinear",
        class_weight="balanced",
        random_state=SEED,
        max_iter=1000,
    )),
])

param_grid = {
    "logreg__penalty": ["l1", "l2"],
    "logreg__C": [0.1, 1.0, 3.0, 10.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
gs = GridSearchCV(base, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
gs.fit(X_train, y_train)

pipe = gs.best_estimator_          # the tuned liblinear model
print("Best parameters:", gs.best_params_)
print(f"Best cross-validated AUC: {gs.best_score_:.4f}")

## 6. Evaluate on the test set and compare with SGD

Metrics for the tuned model, followed by a logistic regression fitted by stochastic gradient descent (`SGDClassifier`, log loss, L2 penalty, early stopping). Rows with a probability of at least `THRESHOLD` are predicted as class 1 (Production).

- **AUC / Average precision:** ranking quality (average precision is more informative when one class is rare).
- **Precision / Recall / F1:** computed for the Production class.

In [ ]:
def report(proba, pred, y_true):
    return {
        "AUC": roc_auc_score(y_true, proba),
        "Average Precision": average_precision_score(y_true, proba),
        "Accuracy": accuracy_score(y_true, pred),
        "F1-Score": f1_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred),
    }

# Tuned logistic regression
proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= THRESHOLD).astype(int)

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(4, 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion matrix: tuned LR (liblinear)")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, proba)
plt.title("ROC: tuned LR (liblinear)")
plt.tight_layout()
plt.show()

PrecisionRecallDisplay.from_predictions(y_test, proba)
plt.title("Precision-Recall: tuned LR (liblinear)")
plt.tight_layout()
plt.show()

# SGD-trained logistic regression
pipe_sgd = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("sgd", SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=1e-4,                # L2 strength
        learning_rate="optimal",   # adaptive step size
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=5,
        class_weight="balanced",
        random_state=SEED,
        max_iter=2000,
        tol=1e-4,
    )),
])
pipe_sgd.fit(X_train, y_train)

proba_sgd = pipe_sgd.predict_proba(X_test)[:, 1]
pred_sgd = (proba_sgd >= THRESHOLD).astype(int)

comparison = pd.DataFrame(
    [report(proba, pred, y_test), report(proba_sgd, pred_sgd, y_test)],
    index=["LR_tuned_liblinear", "LR_SGD"],
).round(4)
display(comparison)

## 7. Threshold tuning (illustrative)

The default threshold of 0.5 is not always best for F1. The scan below tries thresholds from 0.05 to 0.95 on the tuned model. **The threshold is chosen on the test set, so the F1 shown here is optimistic** (see the limitations).

In [ ]:
thresholds = np.linspace(0.05, 0.95, 19)
scores = [(t, f1_score(y_test, (proba >= t).astype(int))) for t in thresholds]
best_t, best_f1 = max(scores, key=lambda item: item[1])
print(f"Best threshold for F1: {best_t:.2f} | F1 = {best_f1:.4f}")
y_pred_best = (proba >= best_t).astype(int)

## 8. Coefficients of the tuned model

Because the features are standardized, the size of a coefficient shows how strongly a feature moves the prediction: positive values push towards Production (1), negative values towards Changeover (0). Many features are strongly related to each other, so individual coefficients can be unstable; treat the ranking as a hint, not as proof of importance.

In [ ]:
coefficients = pipe.named_steps["logreg"].coef_.ravel()

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": coefficients,
    "abs_coef": np.abs(coefficients),
}).sort_values("abs_coef", ascending=False)

top20 = coef_df.head(20)
display(top20[["feature", "coef"]].reset_index(drop=True))

plt.figure(figsize=(8, 6))
plt.barh(top20["feature"][::-1], top20["coef"][::-1])   # largest at the top
plt.title("Top 20 standardized coefficients: tuned LR (liblinear)")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

## 9. Error type analysis

Breakdown of the confusion matrix of the tuned model at the default threshold (class 1 = Production).

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
total = tn + fp + fn + tp
n_changeover = int((y_test == 0).sum())
n_production = int((y_test == 1).sum())

print("Counts:")
print(f"  TN (true 0, pred 0): correct changeover:         {tn}")
print(f"  FP (true 0, pred 1): changeover as production:   {fp}")
print(f"  FN (true 1, pred 0): production as changeover:   {fn}")
print(f"  TP (true 1, pred 1): correct production:         {tp}")

print("\nRelative to all samples:")
print(f"  FP rate (0→1) over all samples: {fp / total:.4f}")
print(f"  FN rate (1→0) over all samples: {fn / total:.4f}")
print(f"  FP rate (0→1) over true changeover: {fp / n_changeover:.4f}")
print(f"  FN rate (1→0) over true production: {fn / n_production:.4f}")

if fp == 0 and fn == 0:
    print("\nNo errors: all samples are on the diagonal.")
elif fp > fn:
    print("\nMain error type: changeover misclassified as production (FP).")
elif fn > fp:
    print("\nMain error type: production misclassified as changeover (FN).")
else:
    print("\nErrors are balanced: FP and FN occur equally often.")

## 10. Save the results locally

Written to `../outputs/` (not for upload):

- `lr_tuned_coefficients.csv`: all coefficients of the tuned model;
- `lr_tuned_test_predictions.csv`: true label, predicted probability, and predictions at threshold 0.5 and at the F1-optimal threshold.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

coefs_out = OUTPUT_DIR / "lr_tuned_coefficients.csv"
preds_out = OUTPUT_DIR / "lr_tuned_test_predictions.csv"

coef_df.to_csv(coefs_out, index=False)
pd.DataFrame({
    "y_true": y_test,
    "y_proba": proba,
    "y_pred_0p50": pred,
    "y_pred_bestF1": y_pred_best,
}).to_csv(preds_out, index=False)

for path in (coefs_out, preds_out):
    print("Saved:", path)

## 11. Summary and conclusion

Tuning the regularization of the logistic regression leaves the performance essentially at the level of the baseline: the linear model is limited by its linear decision boundary, not by the regularization strength. The SGD variant reaches a similar but slightly lower score.

**Reference results from the author's original run** (the cleaned notebook may give slightly different numbers when re-run; best parameters were `C = 1.0` with an L1 penalty):

| Model | AUC | Accuracy | F1 | Precision | Recall |
|---|---|---|---|---|---|
| Tuned LR (liblinear) | 0.9684 | 0.9137 | 0.8665 | 0.8112 | 0.9298 |
| LR trained with SGD | 0.9560 | 0.9045 | 0.8479 | 0.8147 | 0.8839 |

At the F1-optimal threshold of 0.60 the tuned model reached F1 = 0.8756, but that threshold was selected on the test set.

**Next steps:** choose the threshold on validation data, use a chronological or group-wise split, and compare with the tree-based models (notebooks 04 to 06).